In [1]:
import os
import torch
import itertools
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPTokenizer
from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel, StableDiffusionPipeline
from transformers import CLIPTextModel
from torch.optim import AdamW

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive/CT-Spring-26/GenModels/final

/content/drive/MyDrive/CT-Spring-26/GenModels/final


## Training TI Model

In [ ]:
MODEL_ID = "runwayml/stable-diffusion-v1-5"
PLACEHOLDER = "<zendaya>"
INITIALIZER = "person"
NUM_VECTORS = 4
IMAGE_DIR = "data/images"
CAPTION_DIR = "data/captions"
OUTPUT_DIR = "output/embeddings"
STEPS = 3000
LR = 5e-4
BATCH_SIZE = 1
SAVE_EVERY = 500
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
class TextualInversionDataset(Dataset):
    def __init__(self, image_dir, caption_dir, tokenizer, size=512):
        self.image_dir = Path(image_dir)
        self.caption_dir = Path(caption_dir)
        self.tokenizer = tokenizer
        self.size = size
        self.image_paths = sorted(self.image_dir.glob("*.png"))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        cap_path = self.caption_dir / (img_path.stem + ".txt")
        # Load and normalize image to [-1, 1]
        image = Image.open(img_path).convert("RGB").resize((self.size, self.size))
        image = torch.tensor(
            list(image.getdata()), dtype=torch.float32
        ).reshape(self.size, self.size, 3)
        image = (image / 127.5 - 1.0).permute(2, 0, 1)
        # Tokenize caption
        caption = cap_path.read_text().strip()
        tokens  = self.tokenizer(
            caption,
            padding="max_length",
            truncation=True,
            max_length=self.tokenizer.model_max_length,
            return_tensors="pt",
        ).input_ids.squeeze(0)
        return {"pixel_values": image, "input_ids": tokens}

In [ ]:
print("Loading model...")
tokenizer = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder").to(DEVICE)
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae").to(DEVICE)
unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet").to(DEVICE)
scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

Loading model...


tokenizer_config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

unet/diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

In [ ]:
placeholder_tokens = [PLACEHOLDER] + [f"{PLACEHOLDER}_{i}" for i in range(1, NUM_VECTORS)]
tokenizer.add_tokens(placeholder_tokens)
text_encoder.resize_token_embeddings(len(tokenizer))
init_id      = tokenizer.encode(INITIALIZER, add_special_tokens=False)[0]
init_embed   = text_encoder.get_input_embeddings().weight[init_id].detach()

token_ids    = tokenizer.convert_tokens_to_ids(placeholder_tokens)
with torch.no_grad():
    for tid in token_ids:
        text_encoder.get_input_embeddings().weight[tid] = init_embed.clone()

vae.requires_grad_(False)
unet.requires_grad_(False)
text_encoder.requires_grad_(False)
embedding_layer = text_encoder.get_input_embeddings()
embedding_layer.weight.requires_grad_(True)

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Parameter containing:
tensor([[-0.0012,  0.0368,  0.0221,  ...,  0.0158,  0.0046, -0.0219],
        [ 0.0152,  0.0262, -0.0132,  ..., -0.0037,  0.0002,  0.0121],
        [-0.0154, -0.0131,  0.0065,  ..., -0.0206, -0.0139, -0.0025],
        ...,
        [ 0.0117,  0.0216, -0.0187,  ...,  0.0077, -0.0042, -0.0185],
        [ 0.0117,  0.0216, -0.0187,  ...,  0.0077, -0.0042, -0.0185],
        [ 0.0117,  0.0216, -0.0187,  ...,  0.0077, -0.0042, -0.0185]],
       device='cuda:0', requires_grad=True)

In [ ]:
optimizer = AdamW(
    [embedding_layer.weight],
    lr=LR,
    betas=(0.9, 0.999),
    eps=1e-8,
)

In [ ]:
dataset = TextualInversionDataset(IMAGE_DIR, CAPTION_DIR, tokenizer)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
data_iter = itertools.cycle(dataloader)

In [ ]:
import time
from tqdm.notebook import tqdm

print(f"Training for {STEPS} steps on {len(dataset)} images...")
text_encoder.train()
start_time = time.time()
losses = []

pbar = tqdm(range(1, STEPS + 1), desc="Training Progress")
for step in pbar:
    batch = next(data_iter)
    pixel_values = batch["pixel_values"].to(DEVICE)
    input_ids = batch["input_ids"].to(DEVICE)

    with torch.no_grad():
        latents = vae.encode(pixel_values).latent_dist.sample() * 0.18215

    noise = torch.randn_like(latents)
    timesteps = torch.randint(0, scheduler.config.num_train_timesteps, (latents.shape[0],), device=DEVICE).long()
    noisy = scheduler.add_noise(latents, noise, timesteps)
    encoder_hidden_states = text_encoder(input_ids)[0]

    pred = unet(noisy, timesteps, encoder_hidden_states).sample
    loss = torch.nn.functional.mse_loss(pred, noise)

    loss.backward()

    with torch.no_grad():
        grad = embedding_layer.weight.grad
        mask = torch.zeros_like(grad)
        mask[token_ids] = 1.0
        embedding_layer.weight.grad = grad * mask

    optimizer.step()
    optimizer.zero_grad()

    elapsed_time = time.time() - start_time
    steps_per_second = step / elapsed_time if elapsed_time > 0 else 0
    remaining_steps = STEPS - step
    estimated_remaining_time = remaining_steps / steps_per_second if steps_per_second > 0 else float('inf')
    pbar.set_description(f"Training Progress (Loss: {loss.item():.4f}, Elapsed: {int(elapsed_time)}s, Remaining: {int(estimated_remaining_time)}s)")
    losses.append(loss.item())
    if step % 100 == 0:
      print(f"Step {step} - Loss: {loss.item():.4f}")

    if step % SAVE_EVERY == 0:
        ckpt = os.path.join(OUTPUT_DIR, f"zendaya_step{step}.pt")
        torch.save({tid: embedding_layer.weight[tid].detach().cpu() for tid in token_ids}, ckpt)
        print(f"  Saved checkpoint -> {ckpt}")

Training for 3000 steps on 19 images...


Training Progress:   0%|          | 0/3000 [00:00<?, ?it/s]

Step 100 - Loss: 0.1348
Step 200 - Loss: 0.0336
Step 300 - Loss: 0.0031
Step 400 - Loss: 0.0438
Step 500 - Loss: 0.0189
  Saved checkpoint -> output/embeddings/zendaya_step500.pt
Step 600 - Loss: 0.1361
Step 700 - Loss: 0.5177
Step 800 - Loss: 0.0359
Step 900 - Loss: 0.0143
Step 1000 - Loss: 0.0202
  Saved checkpoint -> output/embeddings/zendaya_step1000.pt
Step 1100 - Loss: 0.5933
Step 1200 - Loss: 0.2722
Step 1300 - Loss: 0.0388
Step 1400 - Loss: 0.3822
Step 1500 - Loss: 0.4797
  Saved checkpoint -> output/embeddings/zendaya_step1500.pt
Step 1600 - Loss: 0.0605
Step 1700 - Loss: 0.1452
Step 1800 - Loss: 0.1437
Step 1900 - Loss: 0.1360
Step 2000 - Loss: 0.1032
  Saved checkpoint -> output/embeddings/zendaya_step2000.pt
Step 2100 - Loss: 0.2020
Step 2200 - Loss: 0.0173
Step 2300 - Loss: 0.0360
Step 2400 - Loss: 0.1700
Step 2500 - Loss: 0.0855
  Saved checkpoint -> output/embeddings/zendaya_step2500.pt
Step 2600 - Loss: 0.1099
Step 2700 - Loss: 0.0047
Step 2800 - Loss: 0.0109
Step 2900 

In [ ]:
final_path = os.path.join(OUTPUT_DIR, "zendaya_final.pt")
torch.save({tid: embedding_layer.weight[tid].detach().cpu() for tid in token_ids}, final_path)
print(f"\nDone. Final embedding saved to {final_path}")


Done. Final embedding saved to output/embeddings/zendaya_final.pt


## Inference

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

MODEL_ID = "runwayml/stable-diffusion-v1-5"
EMBED_PATH = "output/embeddings/zendaya_final.pt"
PLACEHOLDER = "<zendaya>"

pipe = StableDiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
pipe = pipe.to("cuda")

tokenizer = pipe.tokenizer
text_encoder = pipe.text_encoder

placeholder_tokens = [PLACEHOLDER] + [f"{PLACEHOLDER}_{i}" for i in range(1, 4)]
tokenizer.add_tokens(placeholder_tokens)
text_encoder.resize_token_embeddings(len(tokenizer))

saved = torch.load(EMBED_PATH)
with torch.no_grad():
    for tid, vec in saved.items():
        text_encoder.get_input_embeddings().weight[tid] = vec.to("cuda").half()

prompts = [
      "a close-up portrait of <zendaya> person smiling",
      "a close-up portrait of <zendaya> person with curly hair",
      "a close-up portrait of <zendaya> person outdoors in sunlight",
      "a studio headshot of <zendaya> person",
      "a close-up portrait of <zendaya> person with a neutral expression",
  ]

num_images = 3
for p, prompt in enumerate(prompts):
    output = pipe(
        prompt,
        negative_prompt="full body, cropped face, low quality, blurry",
        num_inference_steps=30,
        guidance_scale=7.5,
        num_images_per_prompt=num_images
    )
    for i, image in enumerate(output.images):
      image.save(f"output/generated/result{p}_{i+1}.png")

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

MODEL_ID = "runwayml/stable-diffusion-v1-5"
EMBED_PATH  = "output/embeddings/zendaya_final.pt"
PLACEHOLDER = "<zendaya>"

pipe = StableDiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
pipe = pipe.to("cuda")

tokenizer = pipe.tokenizer
text_encoder = pipe.text_encoder

placeholder_tokens = [PLACEHOLDER] + [f"{PLACEHOLDER}_{i}" for i in range(1, 4)]
tokenizer.add_tokens(placeholder_tokens)
text_encoder.resize_token_embeddings(len(tokenizer))

saved = torch.load(EMBED_PATH)
with torch.no_grad():
    for tid, vec in saved.items():
        text_encoder.get_input_embeddings().weight[tid] = vec.to("cuda").half()

prompts = [
    "a photo of <zendaya> person at the beach, golden hour, photorealistic",
    "a photo of <zendaya> person in a long dress at the red carpet",
    "a photo of <zendaya> person in a red shirt at an office",
    "an image of <zendaya> person on the street, photorealistic",
    "an image of <zendaya> person in a blue skirt"
]

num_images = 3
for p, prompt in enumerate(prompts):
    output = pipe(
        prompt,
        negative_prompt="blurry, bad face, distorted, watermark, cartoon",
        num_inference_steps=30,
        guidance_scale=7.5,
        num_images_per_prompt=num_images
    )
    for i, image in enumerate(output.images):
      image.save(f"output/generated/result{p+5}_{i+1}.png")

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

## Evaluation

In [9]:
!pip install insightface onnxruntime-gpu opencv-python numpy
# onnxruntime (cpu-only) if no GPU:
!pip install onnxruntime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 17.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 47.3 MB/s eta 0:00:00
  Created wheel for insightface: filename=insightface-0.7.3-cp312-cp312-linux_x86_64.whl size=1071490 sha256=68b8777f7ecfe0edce3195e84e3e12e3f2f002ca11a638a13aebbe11a3287195
  Stored in directory: /root/.cache/pip/wheels/73/3c/e2/6d4815e8a8b33a2006554d65ce0d1f973e768f4c7a222fa675
Successfully built insightface
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 39.0 MB/s eta 0:00:00


In [10]:
import os
import cv2
import numpy as np
from pathlib import Path
from insightface.app import FaceAnalysis

In [ ]:
# Setup ArcFace
app = FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0)

def get_embedding(image_path: str) -> np.ndarray | None:
    """Return L2-normalised ArcFace embedding for the largest face in image."""
    img  = cv2.imread(image_path)
    if img is None:
        print(f"  Could not read {image_path}")
        return None

    faces = app.get(img)
    if not faces:
        print(f"  No face detected in {image_path}")
        return None

    face  = max(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))
    embed = face.embedding
    return embed / np.linalg.norm(embed)

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b))

download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:10<00:00, 26134.08KB/s]
/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:149: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)


In [ ]:
# Build reference embedding from training images
def build_reference_embedding(image_dir: str) -> np.ndarray:
    """
    Average all training image embeddings into one reference vector.
    More robust than using a single image as reference.
    """
    embeddings = []
    for path in sorted(Path(image_dir).glob("*.png")):
        emb = get_embedding(str(path))
        if emb is not None:
            embeddings.append(emb)

    if not embeddings:
        raise ValueError("No valid faces found in training images")

    mean = np.mean(embeddings, axis=0)
    return mean / np.linalg.norm(mean)

In [ ]:
def evaluate_generated(
    reference_embedding: np.ndarray,
    generated_dir: str,
    threshold: float = 0.45,
) -> dict:
    results = []
    no_face = []
    passing = []
    failing = []

    for path in sorted(Path(generated_dir).glob("*.png")):
        emb = get_embedding(str(path))
        if emb is None:
            no_face.append(path.name)
            continue

        sim = cosine_similarity(reference_embedding, emb)
        results.append({"file": path.name, "similarity": sim})

        if sim >= threshold:
            passing.append(path.name)
        else:
            failing.append(path.name)

    similarities = [r["similarity"] for r in results]

    summary = {
        "total_images" : len(results) + len(no_face),
        "faces_detected" : len(results),
        "no_face_detected" : len(no_face),
        "mean_similarity" : float(np.mean(similarities)) if similarities else 0.0,
        "std_similarity" : float(np.std(similarities)) if similarities else 0.0,
        "min_similarity" : float(np.min(similarities)) if similarities else 0.0,
        "max_similarity" : float(np.max(similarities)) if similarities else 0.0,
        "pass_rate" : len(passing) / len(results) if results else 0.0,
        "passing_images" : passing,
        "failing_images" : failing,
        "no_face_images" : no_face,
        "per_image" : results,
    }
    return summary

In [ ]:
TRAIN_DIR = "data/images"
GEN_DIR   = "output/generated"
THRESHOLD = 0.45

print("Building reference embedding from training images...")
reference = build_reference_embedding(TRAIN_DIR)

print("Evaluating generated images...")
results = evaluate_generated(reference, GEN_DIR, threshold=THRESHOLD)

print(f"""
── ArcFace Evaluation Results ──────────────────────────────
Total images       : {results['total_images']}
Faces detected     : {results['faces_detected']}
No face detected   : {results['no_face_detected']}

Mean similarity    : {results['mean_similarity']:.4f}
Std deviation      : {results['std_similarity']:.4f}
Min / Max          : {results['min_similarity']:.4f} / {results['max_similarity']:.4f}
Pass rate (≥{THRESHOLD}) : {results['pass_rate']:.1%}

Passing images     : {results['passing_images']}
Failing images     : {results['failing_images']}
────────────────────────────────────────────────────────────
""")

Building reference embedding from training images...
Evaluating generated images...
  No face detected in output/generated/result0_3.png
  No face detected in output/generated/result4_3.png
  No face detected in output/generated/result5_1.png
  No face detected in output/generated/result6_2.png
  No face detected in output/generated/result7_1.png
  No face detected in output/generated/result8_2.png

── ArcFace Evaluation Results ──────────────────────────────
Total images       : 30
Faces detected     : 24
No face detected   : 6

Mean similarity    : 0.2141
Std deviation      : 0.1316
Min / Max          : -0.0161 / 0.5242
Pass rate (≥0.45) : 4.2%

Passing images     : ['result8_1.png']
Failing images     : ['result0_1.png', 'result0_2.png', 'result1_1.png', 'result1_2.png', 'result1_3.png', 'result2_1.png', 'result2_2.png', 'result2_3.png', 'result3_1.png', 'result3_2.png', 'result3_3.png', 'result4_1.png', 'result4_2.png', 'result5_2.png', 'result5_3.png', 'result6_1.png', 'result6_3.

CLIP

In [ ]:
import os
import torch
import numpy as np
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from sklearn.metrics.pairwise import cosine_similarity

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(DEVICE)

clip_processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

prompts = [
    "a close-up portrait of <zendaya> person smiling",
    "a close-up portrait of <zendaya> person with curly hair",
    "a close-up portrait of <zendaya> person outdoors in sunlight",
    "a studio headshot of <zendaya> person",
    "a close-up portrait of <zendaya> person with a neutral expression",
    "a photo of <zendaya> person at the beach, golden hour, photorealistic",
    "a photo of <zendaya> person in a long dress at the red carpet",
    "a photo of <zendaya> person in a red shirt at an office",
    "an image of <zendaya> person on the street, photorealistic",
    "an image of <zendaya> person in a blue skirt"
]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def get_text_embedding(text):
    inputs = clip_processor(
        text=[text],
        return_tensors="pt",
        padding=True
    ).to(DEVICE)
    with torch.no_grad():
        text_features_output = clip_model.get_text_features(**inputs)
        text_features = text_features_output.pooler_output
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)
    return text_features.cpu().numpy()


def get_image_embedding(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = clip_processor(
        images=image,
        return_tensors="pt"
    ).to(DEVICE)
    with torch.no_grad():
        image_features_output = clip_model.get_image_features(**inputs)
        image_features = image_features_output.pooler_output
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)
    return image_features.cpu().numpy()

In [46]:
def evaluate_clip_similarity(generated_dir, prompts):
    scores = []
    for p_idx, prompt in enumerate(prompts):
        text_emb = get_text_embedding(prompt)
        for i in range(3):
            image_name = f"result{p_idx}_{i+1}.png"
            image_path = os.path.join(generated_dir, image_name)
            if not os.path.exists(image_path):
                print(f"Missing: {image_path}")
                continue
            image_emb = get_image_embedding(image_path)
            similarity = cosine_similarity(image_emb, text_emb)[0][0]
            scores.append({
                "image": image_name,
                "prompt": prompt,
                "clip_score": float(similarity)
            })
            print(f"{image_name}: {similarity:.4f}")
    avg_score = np.mean([s["clip_score"] for s in scores])
    return {
        "average_clip_score": float(avg_score),
        "all_scores": scores
    }

results = evaluate_clip_similarity(
    "output/generated",
    prompts
)

print("\nAverage CLIP Score:", results["average_clip_score"])

result0_1.png: 0.3658
result0_2.png: 0.3177
result0_3.png: 0.3176
result1_1.png: 0.3247
result1_2.png: 0.3090
result1_3.png: 0.3180
result2_1.png: 0.3212
result2_2.png: 0.3178
result2_3.png: 0.3472
result3_1.png: 0.3503
result3_2.png: 0.3457
result3_3.png: 0.2975
result4_1.png: 0.2956
result4_2.png: 0.3400
result4_3.png: 0.3292
result5_1.png: 0.2808
result5_2.png: 0.3405
result5_3.png: 0.3031
result6_1.png: 0.3561
result6_2.png: 0.3061
result6_3.png: 0.3093
result7_1.png: 0.3144
result7_2.png: 0.3129
result7_3.png: 0.3402
result8_1.png: 0.3117
result8_2.png: 0.2698
result8_3.png: 0.2973
result9_1.png: 0.3683
result9_2.png: 0.3458
result9_3.png: 0.3463

Average CLIP Score: 0.3233264168103536
